# Pneumonia Ontology Paper — Full Pipeline
**Paper:** *Cross-Dataset Pneumonia Detection Failure as an Ontological Problem: A Semantic Label Alignment Study*

Everything in one notebook. Run top to bottom each session.
- Setup: GPU, Drive, Kaggle, dependencies, GitHub clone
- Phase 1: Train CheXpert model
- Phase 2: Semantic distance matrix (BioMedBERT)
- Phase 3: Zero-shot transfer baseline
- Phase 4: Label confusion / clinical dialect analysis
- Phase 5: Semantic distance vs performance correlation
- Phase 6: Ontology-guided fine-tuning vs standard
- Phase 7: Recovery rate + alpha ablation


## ── SETUP ──────────────────────────────────────────────────────────────

In [ ]:
# S1. GPU CHECK
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Runtime > Change runtime type > GPU')
print(result.stdout)
print(f'PyTorch sees GPU: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')

In [ ]:
# S2. MOUNT GOOGLE DRIVE + PATHS
from google.colab import drive
import os
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/pneumonia_research'   # Drive: results persist
RESULTS_DIR  = f'{BASE_DIR}/results'
DATA_DIR     = '/content/data'                                # Local: raw images
KAGGLE_DIR   = f'{DATA_DIR}/kaggle'
CHEXPERT_DIR = f'{DATA_DIR}/chexpert'

for d in [BASE_DIR, RESULTS_DIR, DATA_DIR, KAGGLE_DIR, CHEXPERT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'  Results  -> {RESULTS_DIR}  (persists across sessions)')
print(f'  Images   -> {DATA_DIR}     (local, re-downloads each session)')

In [ ]:
# S3. KAGGLE CREDENTIALS  <-- paste your username and key here
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'   # <-- replace
os.environ['KAGGLE_KEY']      = 'YOUR_KAGGLE_API_KEY'    # <-- replace
print('Kaggle credentials set.')

In [ ]:
# S4. INSTALL DEPENDENCIES
!pip install -q timm transformers grad-cam scikit-learn pandas \
    matplotlib seaborn tqdm Pillow numpy scipy netcal torchmetrics
print('Dependencies installed.')

In [ ]:
# S5. DOWNLOAD DATASETS TO LOCAL DISK
# Kaggle ~1.2GB (~2 mins), CheXpert ~14GB (~15 mins)
# Re-runs each session since local disk resets — only models/results go to Drive
from pathlib import Path

def count_images(path):
    if not os.path.exists(path): return 0
    return sum(1 for p in Path(path).rglob('*')
               if p.suffix.lower() in {'.jpg','.jpeg','.png'})

if count_images(KAGGLE_DIR) < 1000:
    print('Downloading Kaggle Chest X-Ray (~1.2GB)...')
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia \
        -p {KAGGLE_DIR} --unzip -q
    print(f'Kaggle done: {count_images(KAGGLE_DIR):,} images.')
else:
    print(f'Kaggle ready: {count_images(KAGGLE_DIR):,} images.')

if count_images(CHEXPERT_DIR) < 10000:
    print('Downloading CheXpert (~14GB, ~15 mins)...')
    !kaggle datasets download -d ashery/chexpert \
        -p {CHEXPERT_DIR} --unzip -q
    print(f'CheXpert done: {count_images(CHEXPERT_DIR):,} images.')
else:
    print(f'CheXpert ready: {count_images(CHEXPERT_DIR):,} images.')

In [ ]:
# S6. CLONE GITHUB REPO
import sys
GITHUB_REPO = 'https://github.com/Vybh/pneumonia-ontology-paper'
REPO_DIR    = '/content/pneumonia_research'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    !cd {REPO_DIR} && git pull -q
else:
    !git clone -q {GITHUB_REPO} {REPO_DIR}

sys.path.insert(0, REPO_DIR)
print('Repo ready. src/ modules importable.')

In [ ]:
# S7. GLOBAL CONFIG + IMPORTS
import json, copy, random
import numpy as np
import matplotlib.pyplot as plt

CONFIG = {
    'base_dir':     BASE_DIR,
    'kaggle_dir':   KAGGLE_DIR,
    'chexpert_dir': CHEXPERT_DIR,
    'results_dir':  RESULTS_DIR,
    'chexpert_target_labels': [
        'Pneumonia','Lung Opacity','Consolidation',
        'Edema','Pleural Effusion','Atelectasis'
    ],
    'image_size':   224,
    'batch_size':   32,
    'num_workers':  4,
    'epochs':       20,
    'lr':           1e-4,
    'seed':         42,
    'biomed_model': 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
}

with open(f'{BASE_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
torch.backends.cudnn.deterministic = True

print(f'Device: {DEVICE}')
print('Config saved. Setup complete.')

## ── PHASE 1: Train CheXpert Model ─────────────────────────────────────

ResNet50 multi-label classifier trained on 6 CheXpert labels with label smoothing for uncertain labels.
Checkpoint saved to Drive — only trains once.

In [ ]:
from src.data.datasets import get_chexpert_loaders
from src.models.classifier import CheXpertClassifier
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

chexpert_loaders = get_chexpert_loaders(
    chexpert_root=CONFIG['chexpert_dir'],
    image_size=CONFIG['image_size'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    uncertain_strategy='smooth'
)
print('CheXpert loaders ready.')

In [ ]:
CHEXPERT_CKPT = f'{RESULTS_DIR}/chexpert_model.pt'
os.makedirs(RESULTS_DIR, exist_ok=True)

if os.path.exists(CHEXPERT_CKPT):
    print('Loading saved CheXpert model from Drive...')
    chexpert_model = CheXpertClassifier.load(CHEXPERT_CKPT).to(DEVICE)
else:
    print('Training CheXpert model from scratch...')
    chexpert_model = CheXpertClassifier(num_labels=6, pretrained=True).to(DEVICE)
    optimizer = optim.AdamW(chexpert_model.parameters(),
                            lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(CONFIG['epochs']):
        chexpert_model.train()
        losses = []
        for batch in tqdm(chexpert_loaders['train'],
                          desc=f'Epoch {epoch+1}/{CONFIG["epochs"]}'):
            if batch is None: continue
            imgs, labels, _ = batch
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(chexpert_model(imgs), labels)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        scheduler.step()
        print(f'  Epoch {epoch+1}: loss={np.mean(losses):.4f}')

    chexpert_model.save(CHEXPERT_CKPT)
    print(f'Model saved to Drive: {CHEXPERT_CKPT}')

## ── PHASE 2: Semantic Distance Matrix ─────────────────────────────────

Encode all CheXpert and Kaggle label descriptions using BioMedBERT.
Compute pairwise cosine similarity and distance matrices.
**Produces Figure 1 and Figure 2.**

In [ ]:
from src.analysis.semantic_alignment import BioMedBERTEncoder, SemanticDistanceMatrix

SEMANTIC_DIR = f'{RESULTS_DIR}/semantic'
os.makedirs(SEMANTIC_DIR, exist_ok=True)

if os.path.exists(f'{SEMANTIC_DIR}/similarity_matrix.npy'):
    print('Loading precomputed semantic matrices from Drive...')
    distance_matrix = SemanticDistanceMatrix().load(SEMANTIC_DIR)
else:
    print('Computing BioMedBERT embeddings (~2 mins, runs once)...')
    encoder = BioMedBERTEncoder(device=DEVICE)
    distance_matrix = SemanticDistanceMatrix(encoder=encoder)
    distance_matrix.compute(save_path=SEMANTIC_DIR)

print('\nSemantic distances: CheXpert labels -> Kaggle/Pneumonia')
print(f'  {"Label":<35} {"Distance":>10}')
print('  ' + '-'*47)
for label, dist in distance_matrix.get_chexpert_to_kaggle_distances().items():
    print(f'  {label:<35} {dist:>10.4f}')

In [ ]:
# Figure 1 — Similarity heatmap
fig1 = distance_matrix.plot_heatmap(
    save_path=f'{SEMANTIC_DIR}/fig1_similarity_heatmap.png')
plt.show()
print('Figure 1 saved.')

In [ ]:
# Figure 2 — Distance bar chart
fig2 = distance_matrix.plot_distance_bar(
    save_path=f'{SEMANTIC_DIR}/fig2_distance_bar.png')
plt.show()
print('Figure 2 saved.')

## ── PHASE 3: Zero-Shot Transfer Baseline ──────────────────────────────

Run CheXpert-trained model on Kaggle test set with zero adaptation. Establishes the degradation baseline.

In [ ]:
from src.data.datasets import get_kaggle_loaders
from src.models.classifier import MultiOutputWrapper
from sklearn.metrics import roc_auc_score, f1_score, classification_report

kaggle_loaders = get_kaggle_loaders(
    kaggle_root=CONFIG['kaggle_dir'],
    image_size=CONFIG['image_size'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)

wrapped = MultiOutputWrapper(chexpert_model).to(DEVICE)
wrapped.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels, _ in tqdm(kaggle_loaders['test'], desc='Zero-shot eval'):
        probs = wrapped.predict_pneumonia_prob(imgs.to(DEVICE)).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
preds      = (all_probs >= 0.5).astype(int)

baseline_metrics = {
    'auc': roc_auc_score(all_labels, all_probs),
    'f1':  f1_score(all_labels, preds, zero_division=0),
}

print('=== ZERO-SHOT BASELINE ===')
print(f'  AUC-ROC : {baseline_metrics["auc"]:.4f}')
print(f'  F1      : {baseline_metrics["f1"]:.4f}')
print()
print(classification_report(all_labels, preds,
      target_names=['Normal','Pneumonia']))

## ── PHASE 4: Label Confusion Analysis ─────────────────────────────────

The clinical dialect finding.
When the model misses a Kaggle pneumonia case, what CheXpert label does it fire on instead?
**Produces Figure 3 and Table 2.**

In [ ]:
from src.analysis.label_confusion import (
    PredictionCollector, FailureTaxonomy, compute_label_confusion_matrix
)

CONFUSION_DIR = f'{RESULTS_DIR}/label_confusion'
os.makedirs(CONFUSION_DIR, exist_ok=True)

print('Collecting full multi-label predictions on Kaggle test set...')
collector      = PredictionCollector(chexpert_model, device=DEVICE)
predictions_df = collector.collect(kaggle_loaders['test'])
predictions_df.to_csv(f'{CONFUSION_DIR}/predictions.csv', index=False)
print(f'Collected {len(predictions_df):,} predictions.')
predictions_df.head()

In [ ]:
# Failure taxonomy — Figure 3a
taxonomy = FailureTaxonomy(predictions_df)
taxonomy.analyze().report()

fig3a = taxonomy.plot_taxonomy_pie(
    save_path=f'{CONFUSION_DIR}/fig3a_failure_taxonomy.png')
plt.show()

In [ ]:
# Label activation heatmap — Figure 3b
fig3b = taxonomy.plot_label_activation_heatmap(
    save_path=f'{CONFUSION_DIR}/fig3b_activation_heatmap.png')
plt.show()

In [ ]:
# Label confusion matrix — Table 2
confusion_df, fig_t2 = compute_label_confusion_matrix(
    predictions_df,
    save_path=f'{CONFUSION_DIR}/table2_label_confusion.png')
plt.show()
print(confusion_df.round(1).to_string())

## ── PHASE 5: Semantic Distance vs Performance Correlation ─────────────

Core empirical claim of the paper:
transfer performance correlates with semantic label distance, not just imaging domain shift.
**Produces Figure 4 and Table 3.**

In [ ]:
from src.analysis.semantic_alignment import correlate_distance_with_performance

CORR_DIR = f'{RESULTS_DIR}/correlation'
os.makedirs(CORR_DIR, exist_ok=True)

chexpert_labels = CONFIG['chexpert_target_labels']
label_performance = {}

print(f'  {"Label":<28} {"AUC":>8} {"F1":>8}')
print('  ' + '-'*46)
for label in chexpert_labels:
    col = f'prob_{label}'
    if col in predictions_df.columns:
        probs = predictions_df[col].values
        true  = predictions_df['true_label'].values
        try:
            auc = roc_auc_score(true, probs)
        except:
            auc = 0.5
        f1 = f1_score(true, (probs >= 0.5).astype(int), zero_division=0)
        label_performance[f'CheXpert/{label}'] = {'auc': auc, 'f1': f1}
        print(f'  {label:<28} {auc:>8.4f} {f1:>8.4f}')

correlation_results, fig4 = correlate_distance_with_performance(
    distance_matrix, label_performance,
    save_path=f'{CORR_DIR}/fig4_distance_vs_performance.png')
plt.show()

with open(f'{CORR_DIR}/correlation_results.json', 'w') as f:
    json.dump(correlation_results, f, indent=2)
print('Correlation results saved.')

## ── PHASE 6: Ontology-Guided Fine-Tuning ──────────────────────────────

Proposed fix: re-weight the loss using semantic similarity scores.
Labels ontologically close to Kaggle/Pneumonia get lower penalty.
Compare standard BCE vs OGFT head to head.
**Produces Figure 5 and Table 4.**

In [ ]:
from src.models.classifier import KaggleAdaptedClassifier
from src.training.ontology_finetuning import (
    OntologyGuidedLoss, OntologyFineTuner,
    compute_ontology_weights, compute_recovery_rate
)

FINETUNE_DIR = f'{RESULTS_DIR}/finetuning'
os.makedirs(FINETUNE_DIR, exist_ok=True)

ALPHA = 1.0
ontology_weights = compute_ontology_weights(
    distance_matrix, chexpert_labels,
    target_label='Kaggle/Pneumonia',
    alpha=ALPHA
)

In [ ]:
# Standard BCE fine-tuning
print('--- Standard BCE Fine-Tuning ---')
model_standard = KaggleAdaptedClassifier(
    chexpert_model=copy.deepcopy(chexpert_model)).to(DEVICE)

tuner_std = OntologyFineTuner(model_standard, device=DEVICE)
opt_std   = optim.AdamW(model_standard.parameters(), lr=1e-4)
bce_loss  = nn.BCEWithLogitsLoss()

history_std = tuner_std.finetune(
    kaggle_loaders['train'], kaggle_loaders['val'],
    loss_fn=lambda logits, targets, paths: bce_loss(logits, targets.float()),
    optimizer=opt_std, epochs=10, label='standard')

torch.save(model_standard.state_dict(), f'{FINETUNE_DIR}/model_standard.pt')
print('Standard model saved to Drive.')

In [ ]:
# Ontology-guided fine-tuning
print('--- Ontology-Guided Fine-Tuning (Ours) ---')
model_ogft = KaggleAdaptedClassifier(
    chexpert_model=copy.deepcopy(chexpert_model)).to(DEVICE)

ogft_loss  = OntologyGuidedLoss(
    label_weights={'Pneumonia': ontology_weights['Pneumonia']},
    alpha=ALPHA).to(DEVICE)
opt_ogft   = optim.AdamW(model_ogft.parameters(), lr=1e-4)
tuner_ogft = OntologyFineTuner(model_ogft, device=DEVICE)

history_ogft = tuner_ogft.finetune(
    kaggle_loaders['train'], kaggle_loaders['val'],
    loss_fn=lambda logits, targets, paths: ogft_loss(
        logits.unsqueeze(-1), targets.float().unsqueeze(-1)),
    optimizer=opt_ogft, epochs=10, label='ontology')

torch.save(model_ogft.state_dict(), f'{FINETUNE_DIR}/model_ogft.pt')
print('OGFT model saved to Drive.')

## ── PHASE 7: Final Results & Recovery Rate ────────────────────────────

Evaluate both models on Kaggle test set. Compute recovery rate. Run alpha ablation.

In [ ]:
evaluator = OntologyFineTuner(model_standard, device=DEVICE)
std_metrics  = evaluator.evaluate(kaggle_loaders['test'])
evaluator.model = model_ogft
ogft_metrics = evaluator.evaluate(kaggle_loaders['test'])

print(f'\n{"Method":<28} {"AUC":>8} {"F1":>8} {"Acc":>8} {"ECE":>8}')
print('-' * 58)
print(f'{"Zero-shot (baseline)":<28} '
      f'{baseline_metrics["auc"]:>8.4f} {baseline_metrics["f1"]:>8.4f}')
print(f'{"Standard fine-tuning":<28} '
      f'{std_metrics["auc"]:>8.4f} {std_metrics["f1"]:>8.4f} '
      f'{std_metrics["accuracy"]:>8.4f} {std_metrics["ece"]:>8.4f}')
print(f'{"OGFT (ours)":<28} '
      f'{ogft_metrics["auc"]:>8.4f} {ogft_metrics["f1"]:>8.4f} '
      f'{ogft_metrics["accuracy"]:>8.4f} {ogft_metrics["ece"]:>8.4f}')

recovery = compute_recovery_rate(
    baseline_metrics, std_metrics, ogft_metrics, metric='auc')

# Figure 5 — training curves
tuner_std.history = {'standard': history_std, 'ontology': history_ogft}
fig5 = tuner_std.plot_comparison(
    save_path=f'{RESULTS_DIR}/fig5_finetuning_curves.png')
plt.show()

final = {
    'baseline':   baseline_metrics,
    'standard':   std_metrics,
    'ogft':       ogft_metrics,
    'recovery':   recovery,
    'correlation': correlation_results,
    'alpha':      ALPHA,
}
with open(f'{RESULTS_DIR}/final_results.json', 'w') as f:
    json.dump(final, f, indent=2)
print('All results saved to Drive.')

In [ ]:
# Alpha ablation study
print('Running alpha ablation study...')
alphas   = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
ablation = []

for alpha in alphas:
    m = KaggleAdaptedClassifier(
        chexpert_model=copy.deepcopy(chexpert_model)).to(DEVICE)
    w = compute_ontology_weights(
        distance_matrix, chexpert_labels,
        target_label='Kaggle/Pneumonia', alpha=alpha)
    l = OntologyGuidedLoss(
        label_weights={'Pneumonia': w['Pneumonia']}, alpha=alpha).to(DEVICE)
    t = OntologyFineTuner(m, device=DEVICE)
    o = optim.AdamW(m.parameters(), lr=1e-4)
    t.finetune(
        kaggle_loaders['train'], kaggle_loaders['val'],
        loss_fn=lambda logits, targets, paths: l(
            logits.unsqueeze(-1), targets.float().unsqueeze(-1)),
        optimizer=o, epochs=5, label='ontology')
    met = t.evaluate(kaggle_loaders['test'])
    ablation.append({'alpha': alpha, **met})
    print(f'  alpha={alpha}: AUC={met["auc"]:.4f}  F1={met["f1"]:.4f}')

fig_abl, ax = plt.subplots(figsize=(8, 5))
ax.plot([r['alpha'] for r in ablation],
        [r['auc']   for r in ablation],
        'o-', lw=2.5, ms=8, color='#FF5722', label='OGFT')
ax.axhline(baseline_metrics['auc'], color='gray', ls='--', label='Zero-shot baseline')
ax.axhline(std_metrics['auc'],      color='blue', ls='--', label='Standard fine-tuning')
ax.set_xlabel('Alpha (ontological weighting strength)', fontsize=12)
ax.set_ylabel('AUC-ROC on Kaggle Test Set', fontsize=12)
ax.set_title('Alpha Ablation Study\nEffect of Ontological Weighting Strength',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig_ablation_alpha.png', dpi=300, bbox_inches='tight')
plt.show()
print('Ablation complete. All figures saved to Drive.')